# Prompt Optimization with LangMem

## What is Prompt Optimization?

Prompt optimization is the process of **automatically improving a system prompt** by analyzing past conversations and their outcomes. Instead of manually tweaking prompts by trial and error, you provide the optimizer with:

1. **Conversation histories** — what the user asked and what the assistant replied
2. **Annotations** — optional feedback about what was good or bad (scores, comments, revised answers)

The optimizer studies these examples and rewrites your prompt to produce better future responses.

## `create_prompt_optimizer` (SinglePromptOptimizer)

LangMem provides `create_prompt_optimizer` to optimize **a single system prompt**. It supports three distinct optimization strategies (`kind`):

| Strategy | LLM Calls | Speed | Quality | Best For |
|---|---|---|---|---|
| `prompt_memory` | ~1 | Fastest | Good | Quick iterations, token budget is tight |
| `metaprompt` | 1–5 | Fast | Better | Most use cases, good quality/cost balance |
| `gradient` | 2–10 | Slowest | Best | High-stakes prompts where quality > speed |

> **When to use `MultiPromptOptimizer` instead?** Use `create_multi_prompt_optimizer` when your agent has **multiple prompts** (e.g., a router prompt + a responder prompt). It handles "credit assignment" — figuring out *which* prompt caused a failure — so each prompt is optimized independently but coherently. For a single system prompt, `create_prompt_optimizer` is all you need.

## Setup

Import `create_prompt_optimizer` from `langmem`. Make sure your LLM API key is set in your environment (or `.env` file).

In [7]:
from langmem import create_prompt_optimizer

## Step 1: Define Trajectories (Training Data)

A **trajectory** is a tuple of `(conversation, annotation)` where:

- **`conversation`** — a list of `{"role": ..., "content": ...}` messages (the same format as OpenAI/Anthropic chat)
- **`annotation`** — optional feedback. Can be:
  - `None` — no explicit feedback; the optimizer infers what went wrong from the conversation itself (e.g., the user had to rephrase their question)
  - `{"score": float, "comment": str}` — a numeric score (0–1) with a human-written comment
  - `{"revised": str}` — a corrected/improved version of the assistant's response

Think of trajectories as your **labeled training dataset** — the more and richer examples you provide, the better the optimizer can identify patterns in what the prompt is getting wrong.

In [8]:
# A list of (conversation, annotation) pairs — the optimizer's training examples.
# Each conversation shows a real interaction; the annotation explains what was wrong or good.
trajectories = [
    # --- Example 1: No annotation ---
    # The optimizer infers the problem from context:
    # the user had to follow up asking for moons, which means the first answer was incomplete.
    (
        [
            {"role": "user",      "content": "Tell me about Mars"},
            {"role": "assistant", "content": "Mars is the fourth planet..."},
            {"role": "user",      "content": "I wanted more about its moons"},  # implicit signal: answer was lacking
        ],
        None,  # no explicit annotation; optimizer infers dissatisfaction from the follow-up
    ),

    # --- Example 2: Score + comment annotation ---
    # A numeric score (0.0 = terrible, 1.0 = perfect) plus a human comment.
    # Score 0.9 = mostly good, but the comment tells us what to improve.
    (
        [
            {"role": "user",      "content": "What are Mars' moons?"},
            {"role": "assistant", "content": "Mars has two moons: Phobos and Deimos..."},
        ],
        {
            "score": 0.9,
            "comment": "Should include more details and recommended follow-up questions",
        },
    ),

    # --- Example 3: Revised response annotation ---
    # A human (or evaluation system) provides a corrected version of the assistant's response.
    # The optimizer compares what was said vs. what should have been said.
    (
        [
            {"role": "user",      "content": "Compare Mars and Earth"},
            {"role": "assistant", "content": "Mars and Earth have many differences..."},
        ],
        # The original response only mentioned differences; the revision shows similarities matter too.
        {"revised": "Earth and Mars have many similarities and differences..."},
    ),
]

---

## The Three Optimizer Strategies

All three strategies take the same inputs (`trajectories` + `prompt`) and return an improved prompt string. The difference is **how they reason** about the problem:

```
prompt_memory → "What rules can I extract from these examples?"
metaprompt   → "What instructions would produce better outputs? Let me reflect."
gradient     → "What specifically failed? Let me trace back and fix it step by step."
```

---

## Strategy 1: `metaprompt` — Reflective Prompt Rewriting

### How it works

The `metaprompt` strategy uses a **meta-level LLM** (a prompt that tells the LLM how to improve prompts) to rewrite your prompt. It:

1. Reads all trajectories and annotations
2. Reflects on what the current prompt is missing or getting wrong
3. Produces a rewritten prompt in one shot (with optional reflection cycles)

### Config options

- `min_reflection_steps` — minimum number of self-critique passes (0 = can skip if already confident)
- `max_reflection_steps` — caps how many times the LLM re-examines its own rewrite before finalizing

### When to use it

- **Default choice** for most situations
- Good balance between speed (~1–5 LLM calls) and output quality
- Works well when you have clear annotations (scores/comments/revisions)

In [9]:
# Create a metaprompt optimizer.
# The LLM model specified here is used to DO the optimization (not the model being optimized for).
optimizer = create_prompt_optimizer(
    "anthropic:claude-sonnet-4-6",  # the optimizer LLM
    kind="metaprompt",              # reflective rewriting strategy
    config={
        "max_reflection_steps": 1,  # at most 1 self-critique pass (faster)
        "min_reflection_steps": 0,  # can skip reflection if the first draft looks good
    },
)

# Run the optimizer: feed it the trajectories and the current prompt.
# It returns the improved prompt as a plain string.
updated = optimizer.invoke(
    {
        "trajectories": trajectories,
        "prompt": "You are a planetary science expert",  # the prompt to improve
    }
)

print(updated)  # print the new, optimized prompt

You are a planetary science expert. When answering questions, follow these guidelines:

1. **Be thorough and proactive**: Provide comprehensive answers that naturally cover closely related sub-topics a user is likely to care about (e.g., when discussing a planet, briefly address key features such as moons, atmosphere, and surface conditions where relevant).

2. **Balanced comparisons**: When comparing planetary bodies, always address both similarities *and* differences to give a complete picture.

3. **Suggest follow-up questions**: At the end of each response, suggest 2–3 relevant follow-up questions the user might want to explore next, to help guide their learning.


## Strategy 2: `gradient` — Iterative Gradient-Descent-Style Optimization

### How it works

The `gradient` strategy is inspired by **gradient descent in machine learning** — but applied to natural language:

1. It identifies specific "failure signals" from the trajectories (analogous to computing a loss)
2. It proposes targeted edits to the prompt to fix those failures (analogous to a gradient step)
3. It repeats this for `N` reflection cycles, each time building on the previous improvement

This makes it the **most thorough** strategy — it doesn't just rephrase; it traces failures back to specific prompt shortcomings and patches them surgically.

### Config options

- `min_reflection_steps` — at least this many improvement cycles are guaranteed to run
- `max_reflection_steps` — caps the total number of cycles (each cycle = multiple LLM calls)

### When to use it

- When prompt quality is **critical** (customer-facing agents, high-stakes decisions)
- When you have **many trajectories** with rich annotations (gives the optimizer more to work with)
- When you have the **time and token budget** for 2–10 LLM calls per optimization run
- When `metaprompt` results feel too generic or miss subtle failure patterns

In [10]:
# The gradient optimizer runs multiple improvement cycles.
# Each cycle: identify what failed → propose fix → evaluate → repeat.
optimizer = create_prompt_optimizer(
    "anthropic:claude-sonnet-4-6",  # the optimizer LLM
    kind="gradient",                 # iterative gradient-descent-style strategy
    config={
        "max_reflection_steps": 3,  # run at most 3 improvement cycles (more = better but slower)
        "min_reflection_steps": 1,  # always run at least 1 cycle, even if the first draft seems good
    },
)

# Same invocation interface as all other strategies — drop-in replacement
updated = optimizer.invoke(
    {
        "trajectories": trajectories,
        "prompt": "You are a planetary science expert",
    }
)

print(updated)

You are a planetary science expert. When responding:
- Provide detailed, comprehensive answers that cover relevant sub-topics the user is likely to care about.
- When comparing two subjects, address both their similarities and their differences.
- At the end of each response, suggest 2–3 relevant follow-up questions the user might want to explore.


## Strategy 3: `prompt_memory` — Memory-Augmented Rule Extraction

### How it works

The `prompt_memory` strategy treats optimization as a **memory extraction task**:

1. It scans trajectories and extracts **behavioral rules** (e.g., "always suggest follow-up questions", "cover similarities *and* differences")
2. It appends these rules to your existing prompt in a single LLM call

This is conceptually the simplest approach — it's essentially doing: *"What do these conversations tell me I should always/never do?"* and writing those rules down.

### Config options

This strategy has minimal configuration — it intentionally runs in **1 LLM call** with no reflection cycles.

### When to use it

- When you need **fast turnaround** — ideal for real-time or high-frequency optimization loops
- When your token/cost budget is tight
- During early **prototyping** — quickly get a "good enough" prompt before switching to gradient for polish
- When your trajectories are simple and the failure patterns are obvious

In [11]:
# The prompt_memory optimizer extracts behavioral rules from trajectories in a single pass.
# No config needed — it doesn't support reflection cycles (by design, for speed).
optimizer = create_prompt_optimizer(
    "anthropic:claude-sonnet-4-6",  # the optimizer LLM
    kind="prompt_memory",            # single-pass rule-extraction strategy
)

# Same interface — all three strategies are interchangeable drop-in replacements
updated = optimizer.invoke(
    {
        "trajectories": trajectories,
        "prompt": "You are a planetary science expert",
    }
)

print(updated)

You are a planetary science expert. When answering questions:
- Provide comprehensive, detailed responses that anticipate related topics the user might be interested in (e.g., when discussing a planet, consider covering its moons, atmosphere, geology, etc.).
- At the end of your responses, suggest relevant follow-up questions the user might want to explore.
- When comparing planets or celestial bodies, address both their similarities and differences.
- Aim to be thorough and educational while remaining engaging and accessible.


---

## When to Use Which Strategy?

```
Is speed / cost the priority?
  └─ YES → prompt_memory   (1 LLM call, fastest)
  └─ NO  → Is max quality the priority?
              └─ YES → gradient       (2–10 calls, most thorough)
              └─ NO  → metaprompt    (1–5 calls, best default)
```

| Scenario | Recommended Strategy |
|---|---|
| Quick prototype, first iteration | `prompt_memory` |
| General production use | `metaprompt` |
| Customer-facing critical agent | `gradient` |
| Nightly offline optimization job | `gradient` |
| Real-time in-flight optimization | `prompt_memory` |
| You have lots of annotated data | `gradient` |
| You only have implicit signals | `metaprompt` or `prompt_memory` |

### `SinglePromptOptimizer` vs `MultiPromptOptimizer`

| | `create_prompt_optimizer` | `create_multi_prompt_optimizer` |
|---|---|---|
| **Use when** | Your agent has one system prompt | Your agent has multiple prompts (e.g., router + responder) |
| **Credit assignment** | N/A (only one prompt to blame) | Automatically figures out which prompt caused the failure |
| **Input** | Single `prompt` string | List of `Prompt` objects |
| **Optimization strategies** | `metaprompt`, `gradient`, `prompt_memory` | Same strategies, applied per prompt |

> **Rule of thumb:** Start with `create_prompt_optimizer` + `metaprompt`. Upgrade to `gradient` when you need more quality, and switch to `create_multi_prompt_optimizer` only when you have a multi-stage pipeline.

---

## Summary

```python
from langmem import create_prompt_optimizer

# 1. Collect trajectories: (conversation, annotation) pairs
trajectories = [(messages, feedback), ...]

# 2. Create an optimizer — choose your strategy
optimizer = create_prompt_optimizer(
    "anthropic:claude-sonnet-4-6",
    kind="metaprompt",   # or "gradient" or "prompt_memory"
)

# 3. Invoke it — get back an improved prompt string
new_prompt = optimizer.invoke({"trajectories": trajectories, "prompt": "Your current prompt"})
```

**Key takeaways:**
- All three strategies share the same interface — switching is one line change (`kind=...`)
- Trajectories are the core input — richer annotations → better optimization
- `prompt_memory` for speed, `metaprompt` for balance, `gradient` for quality
- Use `create_multi_prompt_optimizer` when you have multiple prompts in a pipeline